# Lab 3–4 · 역기구학과 궤적

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djfksjd/robot_study/blob/main/notebooks/02_inverse_kinematics_trajectory.ipynb)

목표
1. 목표 좌표 → 관절값(IK). 팔꿈치 해 2개(right/left)를 비교한다.
2. 도달 불가·관절 한계 초과를 **거부**하는 것을 확인한다.
3. 관절 공간 보간과 직선(카테시안) 보간의 차이를 그래프로 본다.

In [ ]:
# 0) 저장소 준비 — Colab에서는 처음 한 번 clone, 로컬에서는 저장소 루트에서 실행하면 건너뛴다
import os, sys
if not os.path.exists("model/scara_params.json"):
    if not os.path.exists("robot_study"):
        !git clone -q https://github.com/djfksjd/robot_study.git
    %cd robot_study
sys.path.insert(0, "python")
print("작업 폴더:", os.getcwd())

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scara_kinematics import Scara, joint_trajectory, cartesian_line, min_duration, PARAMS_PATH
robot = Scara.load()

for elbow in ("right", "left"):
    q = robot.ik(300, 150, 15, 0.0, elbow=elbow)
    print(elbow, "θ1={:.1f}° θ2={:.1f}° d3={:.1f}mm θ4={:.1f}°".format(math.degrees(q[0]), math.degrees(q[1]), q[2], math.degrees(q[3])))
    print("   FK 검산 ->", [round(v, 3) for v in robot.fk(q)])

## 1. 거부되어야 하는 목표 — 안전 게이트의 기초

In [ ]:
for target in [(500, 0, 50), (50, 50, 50), (300, 0, -100), (-300, -10, 50)]:
    try:
        robot.ik_best(*target)
        print(target, "-> OK")
    except ValueError as e:
        print(target, "-> 거부:", e)

## 2. 관절 보간 vs 직선 보간 — TCP 경로가 어떻게 다른가

In [ ]:
p0, p1 = (330, 120, 60, 0.0), (60, 330, 60, 0.0)
q0 = robot.ik_best(*p0)
q1 = robot.ik_best(*p1, q_now=q0)
T = min_duration(PARAMS_PATH, q0, q1)
print(f"속도 한계를 지키는 최소 시간 ≈ {T:.2f} s")

jt = joint_trajectory(q0, q1, T, dt=0.02)
ct = cartesian_line(robot, p0, p1, T, dt=0.02, q_now=q0)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
for traj, name in [(jt, "joint interp"), (ct, "cartesian line")]:
    xy = np.array([robot.fk(q)[:2] for _, q in traj])
    axs[0].plot(xy[:, 0], xy[:, 1], label=name)
axs[0].set_aspect("equal"); axs[0].grid(True); axs[0].legend(); axs[0].set_title("TCP path (top view)")
t = [s for s, _ in jt]
for k, lab in [(0, "theta1"), (1, "theta2")]:
    axs[1].plot(t, [math.degrees(q[k]) for _, q in jt], label=f"{lab} joint")
    axs[1].plot(t, [math.degrees(q[k]) for _, q in ct], "--", label=f"{lab} line")
axs[1].grid(True); axs[1].legend(); axs[1].set_xlabel("t (s)"); axs[1].set_ylabel("deg"); axs[1].set_title("joint angles")
plt.tight_layout(); plt.show()

## 3. 픽앤플레이스 한 번의 Z 높이 계획

In [ ]:
z_safe, z_grasp = 90, 15
seq = [("위로 접근", (330, 120, z_safe)), ("내려감", (330, 120, z_grasp)), ("상승", (330, 120, z_safe)),
       ("목표 위", (60, 330, z_safe)), ("내려놓기", (60, 330, z_grasp + 1)), ("상승", (60, 330, z_safe))]
q = (0, 0, 0, 0)
for name, (x, y, z) in seq:
    q = robot.ik_best(x, y, z, 0.0, q_now=q)
    print(f"{name:6s} d3={q[2]:6.1f} mm  θ1={math.degrees(q[0]):7.1f}°  θ2={math.degrees(q[1]):7.1f}°")

## 과제
- 직선 보간이 실패(IK 예외)하는 시작·끝점 쌍을 찾아라. 왜 실패하는가?
- `scara_params.json` 의 `max_speed` 를 절반으로 바꾸면 최소 시간은?